# 02 · The second stage: a cross-encoder re-judges the candidates

The first stage of `text2ipc` embeds the text and the 80k IPC entries separately and
ranks by cosine. It finds the right entries far more often than it puts them first:
with 50 candidates, the office's subclass is in the list 64% of the time but first
only 24% of the time (`docs/evals.md`). This notebook shows the second stage that
closes part of that gap: a **cross-encoder** reads the text and an entry's path
together and gives a relevance verdict, and the two scores are fused.

Prerequisites: the Portuguese index from notebook 01, plus the reranker model, which
`sentence-transformers` downloads on first use (`BAAI/bge-reranker-v2-m3`, 2.2 GB).
On an Apple M-series GPU it judges about 70 pairs per second.

In [1]:
import time
from pathlib import Path

import pandas as pd

from text2ipc import IpcClassifier
from text2ipc.eval import fusion_results, judge_candidates, load_many

ROOT = Path.cwd() if (Path.cwd() / "evals").is_dir() else Path.cwd().parent
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 90)


def as_frame(matches):
    return pd.DataFrame(
        [
            {
                "symbol": m.pretty,
                "score": round(m.score, 3),
                "sim": round(m.similarity, 3),
                "judge": None if m.judge is None else round(m.judge, 2),
                "entry": m.title,
            }
            for m in matches
        ]
    )


pt = IpcClassifier("20260101", lang="PT")
print(pt.version, pt.lang, pt.index.meta.model)

20260101 PT st:intfloat/multilingual-e5-base


## 1. One application, before and after

The abstract of a real application from RPI 2131 (PI 0620431-7), filed by INPI
under A61K 9/70: medicinal preparations in patches for dermal or transdermal
delivery. It also talks about hair care, which pulls the first stage towards
hairdressing (A45D). The eval text is the abstract alone, as here.

In [2]:
text = (
    "A presente invenção refere-se ao uso de poliuretanos formadores de película que "
    "encontram utilização em agentes para o cuidado dos cabelos ou de misturas desses "
    "poliuretanos com outros polimeros em preparações farmacêuticas para a aplicação "
    "dérmica ou transdérmica de substâncias ativas, bem como emplastros e preparações "
    "farmacêuticas contendo esses poliuretanos para o cuidado dos cabelos."
)
first = pt.classify(text, level="subgroup", top_k=10)
as_frame(first)

/Users/jaimenms/code/personal/study-text-to-ipc/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14416.15it/s]

,symbol,score,sim,judge,entry
0,A45D 7/06,0.847,0.847,None,térmicos e químicos combinados
1,A24C 1/20,0.845,0.845,None,Moldes rotativos para feixes de charutos
2,A45D 7/04,0.845,0.845,None,químicos
3,A45D 6/14,0.845,0.845,None,Grampos simples para rolos de cabelo
4,A45D 6/16,0.845,0.845,None,Grampos enroladores
5,D03D 39/22,0.844,0.844,None,"Teares para tecidos felpudos tipo esponja (""terry looms"")"
6,A45D 6/10,0.844,0.844,None,para onduladores planos
7,A61K 47/60,0.844,0.844,None,"o composto orgânico macromolecular sendo um oligômero, polímero ou dendrímero de polio..."


In [3]:
t0 = time.time()
reranked = pt.classify(text, level="subgroup", top_k=10, rerank=True)
print(f"reranked in {time.time() - t0:.1f} s (the first call also loads the model)")
as_frame(reranked)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights:   2%|▏         | 9/393 [00:00<00:34, 10.98it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 476.18it/s]

reranked in 5.3 s (the first call also loads the model)


,symbol,score,sim,judge,entry
0,A61K 47/60,0.456,0.844,0.08,"o composto orgânico macromolecular sendo um oligômero, polímero ou dendrímero de polio..."
1,A45D 7/06,0.443,0.847,0.05,térmicos e químicos combinados
2,A45D 7/04,0.440,0.845,0.04,químicos
3,A45D 7/02,0.438,0.843,0.04,térmicos
4,A45D 2/12,0.436,0.842,0.04,Enroladores ou onduladores de cabelo para serem usados paralelamente ao couro cabeludo...
5,A45D 24/26,0.435,0.842,0.03,"com paredes flexíveis do dispositivo de armazenamento do líquido, pasta ou pó"
6,A45D 2/10,0.435,0.843,0.03,em forma de carretéis ou bobinas
7,A45D 6/18,0.435,0.842,0.03,"Dispositivos de proteção ou envoltórios para rolos de cabelo ou similares, quando em uso"
8,A45D 24/34,0.435,0.844,0.03,Dispositivos para repartir os cabelos no alto da cabeça
9,A45D 19/16,0.434,0.842,0.03,"Tratamento superficial do cabelo por vapor, óleo ou similares"


`score` is now `cosine × (0.5 + 0.5 × judge)`, where `judge` is the cross-encoder's
verdict in 0..1: an entry the judge dislikes keeps at most half its cosine. Scores
across the two tables are therefore on different scales; within one table the order
is what matters. `rerank=True` judged 50 candidates (`max(5 × top_k, 25)`) and
returned the best 10: A61K 47/60 (pharmaceutical carriers) moves to the top, the
hairdressing entries slide down. The verdicts are small (0.08 against 0.05), which
is the next point.

One caveat found while writing this: with the application's title added as a
second paragraph (a long line in capitals), the first stage drifts to other entries
and the judge gives almost every candidate a verdict near 0.99. The cross-encoder is
sensitive to the form of the text; the evals, and this notebook, use the abstract.

## 2. The judge's verdicts

`clf.reranker()` gives the model itself. Its logits are not calibrated: the
offset changes from text to text, so a verdict of 0.1 or 0.9 says little on its
own and unrelated entries can still get a positive logit. What carries the signal
is the order within one query, and the `blend` fusion lets the cosine break the
near-ties the judge leaves.

In [4]:
judge = pt.reranker()
pairs = [(m.pretty, m.title) for m in first[:6]]
logits = judge.score(text, [pt.index.text_of(m.symbol) for m in first[:6]])
pd.DataFrame(
    {"symbol": [p[0] for p in pairs], "logit": logits.round(2), "entry": [p[1] for p in pairs]}
)

,symbol,logit,entry
0,A45D 7/06,-3.04,térmicos e químicos combinados
1,A24C 1/20,-8.51,Moldes rotativos para feixes de charutos
2,A45D 7/04,-3.16,químicos
3,A45D 6/14,-3.82,Grampos simples para rolos de cabelo
4,A45D 6/16,-3.83,Grampos enroladores
5,D03D 39/22,-7.46,"Teares para tecidos felpudos tipo esponja (""terry looms"")"


## 3. Evaluation: one judged run, several fusions

Judging is the expensive part, so `judge_candidates` runs the first stage and the
reranker once per case and keeps the candidates with both scores; `fusion_results`
then evaluates every fusion rule from that one run. 200 cases (100 with abstracts,
100 title-only), up to 25 candidates each (fewer where branches collapse): about
4,000 pairs.

In [5]:
cases = load_many(str(ROOT / "evals" / "rpi_*.jsonl"), sample=200, seed=0)
t0 = time.time()
judged = judge_candidates(pt, cases, candidates=25, level="subgroup")
elapsed = time.time() - t0
pairs = sum(len(j.matches) for j in judged)
rate = pairs / elapsed
print(f"{len(judged)} cases, {pairs} pairs judged in {elapsed:.0f} s ({rate:.0f} pairs/s)")

200 cases, 4019 pairs judged in 59 s (68 pairs/s)


In [6]:
results = fusion_results(judged, top_k=10, level="subgroup")
rows = []
for name, result in results.items():
    for level in ("subclass", "group", "subgroup"):
        rows.append(
            {
                "fusion": name,
                "level": level,
                "hit@1": result.rate(level, 1),
                "hit@3": result.rate(level, 3),
                "hit@10": result.rate(level, 10),
            }
        )
table = pd.DataFrame(rows).pivot(index="level", columns="fusion", values="hit@1")
table = table.loc[["subclass", "group", "subgroup"], ["first", "judge", "product", "blend"]]
table.apply(lambda col: col.map("{:.1%}".format))

fusion,first,judge,product,blend
level,,,,
subclass,23.0%,31.5%,31.5%,31.0%
group,8.0%,17.0%,17.0%,16.0%
subgroup,2.0%,3.0%,2.5%,2.0%


### Reading the table

- `first` is the first stage alone; `judge` orders by the cross-encoder alone;
  `product` is `cosine × judge`; `blend`, the default, is `cosine × (0.5 + 0.5 × judge)`.
- On the full 991-case run (`docs/evals.md`) `blend` takes subclass@1 from 23.6% to
  31.6% and group@1 from 11.2% to 16.6%; 200 cases reproduce the direction with
  wider error bars (about ±3 points at this size).
- The judge decides and the cosine breaks ties: `judge` and `blend` land close, and
  both beat using the judge only as a tie-break (`docs/evals.md`).

In [7]:
pd.DataFrame(rows).pivot(index="level", columns="fusion", values="hit@10").loc[
    ["subclass", "group", "subgroup"], ["first", "judge", "product", "blend"]
].apply(lambda col: col.map("{:.1%}".format))

fusion,first,judge,product,blend
level,,,,
subclass,43.0%,45.0%,45.0%,44.0%
group,23.0%,26.5%,26.5%,25.5%
subgroup,5.0%,6.0%,6.5%,6.5%


## 4. Where the reranker helps, and where it cannot

A case the second stage fixes: the right subclass was in the candidates but not
first. A case it cannot fix: the right entry was not among the candidates at all,
which no reordering can repair. Both are visible by comparing the first-stage order
with the fused order.

In [8]:
from text2ipc.eval import fused_order
from text2ipc.eval.harness import truncate

fixed, lost = [], []
for j in judged:
    gold = {truncate(s, "subclass") for s in j.case.ipc}
    before = [truncate(m.symbol, "subclass") for m in j.matches[:10]]
    after = [truncate(m.symbol, "subclass") for m in fused_order(j, "blend", 10)]
    if before[0] not in gold and after[0] in gold:
        fixed.append(j)
    if not any(s in gold for s in (truncate(m.symbol, "subclass") for m in j.matches)):
        lost.append(j)
print(len(fixed), "cases fixed at subclass@1;", len(lost), "cases with no gold subclass among 25")
j = fixed[0]
print(j.case.id, "gold", j.case.ipc[:2], "|", j.case.text[:110])
pd.DataFrame(
    {
        "first stage": [m.pretty for m in j.matches[:5]],
        "reranked": [m.pretty for m in fused_order(j, "blend", 5)],
        "judge": [round(m.judge, 2) for m in fused_order(j, "blend", 5)],
    }
)

23 cases fixed at subclass@1; 102 cases with no gold subclass among 25
rpi2131:PI0620431-7A2 gold ('A61K0009700000',) | A presente invenção refere-se ao uso de poliuretanos formadores de película que encontram utilização em agente


,first stage,reranked,judge
0,A45D 7/06,A61K 47/60,0.08
1,A24C 1/20,A45D 7/06,0.05
2,A45D 7/04,A45D 7/04,0.04
3,A45D 6/14,A45D 7/02,0.04
4,A45D 6/16,A45D 24/34,0.03


## 5. Cost and where it runs

- Package: `t2ipc classify --rerank`, `t2ipc eval --rerank`, or `rerank=True`. The
  model is downloaded once; the cross-encoder adds well under a second per query on
  a GPU and a few seconds on CPU.
- Endpoint: `{"parameters": {"rerank": true}}` on the Hugging Face handler.
- Browser: the Space runs `jina-reranker-v2-base-multilingual` in 8 bits inside a
  Web Worker, about one second per candidate; on 300 cases it equals the bge model
  at @1 (`docs/evals.md`).
- Ceiling: with 50 candidates the subclass is present 64% of the time; the reranker
  reaches 32% at rank 1. The next gains are in the first stage (a stronger embedder)
  or in a reranker trained on patent data.